# Observed genotype variance vs 2p(1-p), by allele frequency, on the GRM panel
Ratio = 1 + F: above 1 means fewer heterozygotes than HWE (structure, inbreeding), below 1 more.

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
STAGE = "allele_variance"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
RUN = needs_run(OUT, FORCE)
setenv(RUN=int(RUN), BED=single_target(cfg, "grm_panel"), OUT=OUT, THREADS=THREADS, PLINK2=cfg.tool("plink2"))

In [ ]:
def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
%%bash
set -euo pipefail
if [ "$RUN" != "1" ]; then echo "exists, keeping it (FORCE = True to remake)"; exit 0; fi
mkdir -p "$(dirname "$OUT")"
"$PLINK2" --bfile "${BED%.bed}" --geno-counts --threads "$THREADS" --silent --out "${OUT%.gcount}"

In [ ]:
if RUN:
    mark_done(STAGE, OUT, tools={"plink2": "--geno-counts"})
g = pd.read_csv(OUT, sep="\t")
n0, n1, n2 = g["HOM_REF_CT"], g["HET_REF_ALT_CTS"], g["TWO_ALT_GENO_CTS"]
n = n0 + n1 + n2
p = (n1 + 2 * n2) / (2 * n)
s = pd.DataFrame({"maf": np.minimum(p, 1 - p), "exp": 2 * p * (1 - p), "var": (n1 + 4 * n2) / n - (2 * p) ** 2, "het": n1 / n})
s["var"] *= n / (n - 1)
s = s.assign(var_ratio=s["var"] / s["exp"], het_ratio=s["het"] / s["exp"])
rep.count("n_snps", len(s)); rep.note(hwe_filter=cfg.params.get("grm_qc_hwe", cfg.param("qc_hwe")), maf_min=cfg.param("qc_maf"))
MB = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
grp = s.groupby(pd.cut(s["maf"], MB, include_lowest=True), observed=False)
tab = pd.DataFrame({"maf_lo": MB[:-1], "maf_hi": MB[1:], "snps": grp.size().to_numpy(), "mean_maf": grp["maf"].mean().to_numpy(),
                    "mean_2pq": grp["exp"].mean().to_numpy(), "mean_var": grp["var"].mean().to_numpy()})
for v in ("var_ratio", "het_ratio"):
    tab[v] = grp[v].mean().to_numpy(); tab[v + "_se"] = (grp[v].std() / np.sqrt(grp.size())).to_numpy()
tab["F"] = 1 - tab["het_ratio"]
display(rep.table(tab, "by_maf", counts=["snps"], caption="per MAF bin: mean over SNPs; se across SNPs"))

In [ ]:
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
fig, ax = plt.subplots(figsize=(6, 4.5))   # notebook only: every SNP
ax.scatter(s["exp"], s["var"], s=1, alpha=0.2, lw=0, color="0.3", rasterized=True)
ax.plot([0, 0.5], [0, 0.5], "k:", lw=0.8)
ax.set(xlabel="2p(1-p)", ylabel="observed genotype variance", title=RUN_ID); plt.show()

t = tab[tab["snps"] >= 21]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(t["mean_2pq"], t["mean_var"], "o-", color="k", ms=4, lw=1.3)
axes[0].plot([0, 0.5], [0, 0.5], ":", color="0.5", lw=0.8)
axes[0].set(xlabel="mean 2p(1-p)", ylabel="mean observed variance", xlim=(0, 0.52), ylim=(0, 0.52), aspect="equal")
for v, m, lab in [("var_ratio", "o", "variance / 2p(1-p)"), ("het_ratio", "s", "heterozygosity / 2p(1-p)")]:
    axes[1].errorbar(t["mean_maf"], t[v], yerr=1.96 * t[v + "_se"], fmt=m + "-", ms=4, lw=1.2, capsize=2, color="k" if m == "o" else "0.55", label=lab)
axes[1].axhline(1, color="0.5", lw=0.8, ls=":")
axes[1].set(xscale="log", xlabel="MAF (bin mean)", ylabel="observed / expected"); axes[1].legend(frameon=False, fontsize=8)
axes[1].set_xticks([0.01, 0.02, 0.05, 0.1, 0.2, 0.5], ["0.01", "0.02", "0.05", "0.1", "0.2", "0.5"]); axes[1].minorticks_off()
fig.suptitle(RUN_ID); fig.tight_layout()
rep.fig(fig, "variance_by_maf", "GRM panel SNPs in MAF bins (21+ SNPs); mean over SNPs, 95% CI across SNPs"); plt.show()
rep.save()